In [1]:
import os
import json
from PIL import Image
from pathlib import Path
import csv

import torch
from torchvision.models import vit_b_16, ViT_B_16_Weights

In [2]:
# Set Device to GPU if available
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"CUDA Available: {torch.cuda.is_available()}")

CUDA Available: True


In [3]:
DATA_FOLDER = "data"
DATASET_FOLDER = os.path.join(DATA_FOLDER,"circo")
ANNOTATION_FILE = os.path.join(DATASET_FOLDER,"annotations.json")
EMBEDDING_FILE = os.path.join(DATASET_FOLDER,"embeddings.csv")

In [4]:
weights = ViT_B_16_Weights.DEFAULT

model = vit_b_16(weights=weights)
model.heads = torch.nn.Identity()
model = model.to(DEVICE)
model.eval()

preprocess = weights.transforms()

In [5]:
def get_vit_embedding(model,img_file_path):
    image = Image.open(img_file_path).convert("RGB")

    x = preprocess(image)
    x = x.unsqueeze(0).to(DEVICE)  # add batch dimension
    
    with torch.no_grad():
        embedding = model(x)

    return embedding

In [6]:
id_embedding_pair = {}

for file in Path(DATASET_FOLDER).iterdir():
    if file.suffix.lower() in [".jpg", ".jpeg"]:
        embedding = get_vit_embedding(model,file).squeeze(0).tolist()
        id_embedding_pair[file.name] = embedding

In [8]:
with open(EMBEDDING_FILE, "w", newline="") as f:
    writer = csv.writer(f)
    
    writer.writerow(["image_id", "embedding"])
    
    for image_id, embedding in id_embedding_pair.items():
        writer.writerow([image_id, embedding])